# Wholebody Pose Estimation using SAPIENS Model

This notebook predicts wholebody poses (body + hands + face) using the SAPIENS model.
It processes videos from the inputs/{data_collection} directory and outputs wholebody keypoints.

In [3]:
import os
import cv2
import numpy as np
import mmcv
import json
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image

# Import the necessary MMPose components
import mmengine
from mmpose.apis import inference_topdown
from mmpose.apis import init_model as init_pose_estimator
from mmpose.structures import merge_data_samples, split_instances
from mmpose.registry import VISUALIZERS

# IMPORTANT: Import mmpretrain to register the ViT model
# import mmpretrain
# import mmpretrain.models

# Import custom helpers
from helpers.definitions import *

## Configuration

Set up the data collection path and other configuration parameters.

In [2]:
# Configuration
data_collection = 'cha/cha_short'  # Change this to match your data collection
show = True  # Show visualizations
draw_bbox = True  # Draw bounding boxes
resolution = (1920, 1080)  # Processing resolution
kpt_thresh = 0.3  # Keypoint confidence threshold

# Create output directories
pose_output_dir = f'pose_outputs/{data_collection}'
vis_output_dir = f'vis_dir/{data_collection}'

os.makedirs(pose_output_dir, exist_ok=True)
os.makedirs(vis_output_dir, exist_ok=True)

# Input data directory
data_dir = f'inputs/{data_collection}'

# Ensure data directory exists
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"Data directory not found: {data_dir}")

## Initialize SAPIENS Model

Initialize the SAPIENS model directly without using the helper function.

In [4]:
# Path to SAPIENS model configuration and weights
SAPIENS_CONFIG = 'configs/wholebody_2d_keypoint/sapiens/sapiens_1b-210e_coco_wholebody-1024x768.py'
SAPIENS_WEIGHTS = 'checkpoints/sapiens/sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727.pth'

# Set visualization parameters
POSE_VIS_RADIUS = 3
POSE_VIS_ALPHA = 0.8
POSE_VIS_LINE_WIDTH = 2
POSE_KPT_THRESHOLD = 0.3

# Initialize SAPIENS model
try:
    print("Initializing SAPIENS model...")
    pose_estimator_sapiens = init_pose_estimator(
        SAPIENS_CONFIG,
        SAPIENS_WEIGHTS,
        device='cuda',
        cfg_options=dict(model=dict(test_cfg=dict(output_heatmaps=False)))
    )
    
    # Configure visualizer
    pose_estimator_sapiens.cfg.visualizer.radius = POSE_VIS_RADIUS
    pose_estimator_sapiens.cfg.visualizer.alpha = POSE_VIS_ALPHA
    pose_estimator_sapiens.cfg.visualizer.line_width = POSE_VIS_LINE_WIDTH
    visualizer_sapiens = VISUALIZERS.build(pose_estimator_sapiens.cfg.visualizer)
    visualizer_sapiens.set_dataset_meta(pose_estimator_sapiens.dataset_meta, skeleton_style='mmpose')
    
    print("SAPIENS model initialized successfully!")
except Exception as e:
    print(f"Error initializing SAPIENS model: {e}")
    import traceback
    traceback.print_exc()

Initializing SAPIENS model...
Error initializing SAPIENS model: 'mmpretrain.VisionTransformer is not in the mmpose::model registry. Please check whether the value of `mmpretrain.VisionTransformer` is correct or it was registered as expected. More details can be found at https://mmengine.readthedocs.io/en/latest/advanced_tutorials/config.html#import-the-custom-module'


Traceback (most recent call last):
  File "C:\Users\aelvy\AppData\Local\Temp\ipykernel_8728\155409334.py", line 14, in <module>
    pose_estimator_sapiens = init_pose_estimator(
                             ^^^^^^^^^^^^^^^^^^^^
  File "c:\users\aelvy\documents\github\mmpose\mmpose\apis\inference.py", line 104, in init_model
    model = build_pose_estimator(config.model)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\users\aelvy\documents\github\mmpose\mmpose\models\builder.py", line 35, in build_pose_estimator
    return POSE_ESTIMATORS.build(cfg)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\mmengine\registry\registry.py", line 570, in build
    return self.build_func(cfg, *args, **kwargs, registry=self)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\mmengine\registry\build_functions.py", line 234, in build_model_from_cfg
    return build_

## Initialize YOLO Model for Person Detection

We'll use YOLOv8 for person detection before applying the SAPIENS pose estimator.

In [ ]:
# Initialize YOLO model for person detection
from ultralytics import YOLO

model_person = YOLO('checkpoints/yolo/yolo11l.pt')
model_person.conf = 0.5  # Confidence threshold

def detect_person(frame):
    """Detect person bounding boxes in a frame using YOLO.
    
    Args:
        frame: Input frame as numpy array
        
    Returns:
        numpy array of person bounding boxes in format [x1, y1, x2, y2]
    """
    results = model_person(frame, classes=0)  # Class 0 is person
    boxes = []
    
    for r in results:
        boxes_tensor = r.boxes.xyxy.cpu()
        confs = r.boxes.conf.cpu()
        
        for box, conf in zip(boxes_tensor, confs):
            if conf > model_person.conf:
                boxes.append(box.numpy())
    
    return np.array(boxes) if boxes else np.array([])

## SAPIENS Pose Estimation Function

Create functions to estimate poses and visualize them.

In [ ]:
def estimate_sapiens_pose(img, bbox, show=False, write_img=None):
    """Estimate wholebody pose using SAPIENS model.
    
    Args:
        img: Input image (path or numpy array)
        bbox: Bounding box coordinates [[x1,y1,x2,y2]]
        show: Whether to show visualization
        write_img: Optional image to draw visualization on
    
    Returns:
        Tuple of (body_instances, left_hand_instances, right_hand_instances)
    """
    # Ensure bbox is in correct format (N,4)
    if isinstance(bbox, np.ndarray):
        bbox = bbox.reshape(1,-1) if bbox.size == 4 else bbox

    # Predict keypoints using SAPIENS wholebody model
    with torch.inference_mode():
        pose_results = inference_topdown(pose_estimator_sapiens, img, bbox)
    
    data_samples = merge_data_samples(pose_results)
    
    # Visualize if requested
    if visualizer_sapiens is not None and show:
        if write_img is None:
            write_img = img
        if isinstance(write_img, str):
            write_img = mmcv.imread(write_img, channel_order='rgb')
        elif isinstance(write_img, np.ndarray):
            write_img = mmcv.bgr2rgb(write_img)

        visualizer_sapiens.add_datasample(
            'result',
            write_img,
            data_sample=data_samples,
            draw_gt=False,
            draw_heatmap=False,
            draw_bbox=True,
            show_kpt_idx=False,
            skeleton_style='mmpose',
            show=show,
            wait_time=0.1,
            kpt_thr=POSE_KPT_THRESHOLD)
    
    # Get the predicted instances from the data sample
    pred_instances = data_samples.get('pred_instances', None)
    
    if pred_instances is None:
        return None, None, None
    
    # Extract body, left hand, and right hand keypoints
    # Based on the COCO-WholeBody dataset format used by SAPIENS
    # Body keypoints are the first 17 points (0-16)
    # Left hand starts at index 91 and has 21 keypoints (91-111)
    # Right hand starts at index 112 and has 21 keypoints (112-132)
    
    # Create a copy of the instances to avoid modifying the original
    body_instances = pred_instances.clone()
    left_hand_instances = pred_instances.clone()
    right_hand_instances = pred_instances.clone()
    
    # Extract body keypoints (first 17 keypoints)
    if pred_instances.keypoints.shape[1] > 17:
        body_keypoints = pred_instances.keypoints[:, :17, :]
        body_keypoint_scores = pred_instances.keypoint_scores[:, :17]
        body_instances.keypoints = body_keypoints
        body_instances.keypoint_scores = body_keypoint_scores
    else:
        body_instances = None
        
    # Extract left hand keypoints (indices 91-111)
    if pred_instances.keypoints.shape[1] > 111:
        left_hand_keypoints = pred_instances.keypoints[:, 91:112, :]
        left_hand_keypoint_scores = pred_instances.keypoint_scores[:, 91:112]
        left_hand_instances.keypoints = left_hand_keypoints
        left_hand_instances.keypoint_scores = left_hand_keypoint_scores
    else:
        left_hand_instances = None
        
    # Extract right hand keypoints (indices 112-132)
    if pred_instances.keypoints.shape[1] > 132:
        right_hand_keypoints = pred_instances.keypoints[:, 112:133, :]
        right_hand_keypoint_scores = pred_instances.keypoint_scores[:, 112:133]
        right_hand_instances.keypoints = right_hand_keypoints
        right_hand_instances.keypoint_scores = right_hand_keypoint_scores
    else:
        right_hand_instances = None
        
    return body_instances, left_hand_instances, right_hand_instances

## Visualization Function

Create a function to visualize the detected whole-body keypoints.

In [ ]:
def visualize_wholebody_pose(frame, body_instances, left_hand_instances, right_hand_instances):
    """Visualize wholebody pose on a frame.
    
    Args:
        frame: Input frame
        body_instances: Body pose instances
        left_hand_instances: Left hand pose instances
        right_hand_instances: Right hand pose instances
        
    Returns:
        Frame with visualized poses
    """
    vis_frame = frame.copy()
    
    # Draw body keypoints
    if body_instances is not None:
        for i in range(len(body_instances.keypoints)):
            keypoints = body_instances.keypoints[i]
            scores = body_instances.keypoint_scores[i]
            
            # Draw body keypoints with confidence > threshold
            for j, (kp, score) in enumerate(zip(keypoints, scores)):
                if score > kpt_thresh:
                    x, y = map(int, kp)
                    cv2.circle(vis_frame, (x, y), 5, (0, 255, 0), -1)
            
            # Draw connections between keypoints (skeleton)
            # COCO skeleton connections
            connections = [
                (0, 1), (1, 2), (2, 3), (3, 4),  # Head to neck to shoulders
                (5, 6),  # Shoulders
                (5, 7), (7, 9),  # Left arm
                (6, 8), (8, 10),  # Right arm
                (5, 11), (6, 12),  # Shoulders to hips
                (11, 12),  # Hips
                (11, 13), (13, 15),  # Left leg
                (12, 14), (14, 16)  # Right leg
            ]
            
            for conn in connections:
                if scores[conn[0]] > kpt_thresh and scores[conn[1]] > kpt_thresh:
                    pt1 = tuple(map(int, keypoints[conn[0]]))
                    pt2 = tuple(map(int, keypoints[conn[1]]))
                    cv2.line(vis_frame, pt1, pt2, (0, 255, 0), 2)
    
    # Draw left hand keypoints
    if left_hand_instances is not None:
        for i in range(len(left_hand_instances.keypoints)):
            keypoints = left_hand_instances.keypoints[i]
            scores = left_hand_instances.keypoint_scores[i]
            
            # Draw keypoints
            for j, (kp, score) in enumerate(zip(keypoints, scores)):
                if score > kpt_thresh:
                    x, y = map(int, kp)
                    cv2.circle(vis_frame, (x, y), 3, (255, 0, 0), -1)
            
            # Hand connections
            finger_connections = [
                # Thumb
                (0, 1), (1, 2), (2, 3), (3, 4),
                # Index finger
                (0, 5), (5, 6), (6, 7), (7, 8),
                # Middle finger
                (0, 9), (9, 10), (10, 11), (11, 12),
                # Ring finger
                (0, 13), (13, 14), (14, 15), (15, 16),
                # Pinky
                (0, 17), (17, 18), (18, 19), (19, 20)
            ]
            
            for conn in finger_connections:
                if scores[conn[0]] > kpt_thresh and scores[conn[1]] > kpt_thresh:
                    pt1 = tuple(map(int, keypoints[conn[0]]))
                    pt2 = tuple(map(int, keypoints[conn[1]]))
                    cv2.line(vis_frame, pt1, pt2, (255, 0, 0), 1)
    
    # Draw right hand keypoints
    if right_hand_instances is not None:
        for i in range(len(right_hand_instances.keypoints)):
            keypoints = right_hand_instances.keypoints[i]
            scores = right_hand_instances.keypoint_scores[i]
            
            # Draw keypoints
            for j, (kp, score) in enumerate(zip(keypoints, scores)):
                if score > kpt_thresh:
                    x, y = map(int, kp)
                    cv2.circle(vis_frame, (x, y), 3, (0, 0, 255), -1)
            
            # Hand connections (same as left hand)
            finger_connections = [
                # Thumb
                (0, 1), (1, 2), (2, 3), (3, 4),
                # Index finger
                (0, 5), (5, 6), (6, 7), (7, 8),
                # Middle finger
                (0, 9), (9, 10), (10, 11), (11, 12),
                # Ring finger
                (0, 13), (13, 14), (14, 15), (15, 16),
                # Pinky
                (0, 17), (17, 18), (18, 19), (19, 20)
            ]
            
            for conn in finger_connections:
                if scores[conn[0]] > kpt_thresh and scores[conn[1]] > kpt_thresh:
                    pt1 = tuple(map(int, keypoints[conn[0]]))
                    pt2 = tuple(map(int, keypoints[conn[1]]))
                    cv2.line(vis_frame, pt1, pt2, (0, 0, 255), 1)
    
    return vis_frame

## Process Videos

Function to process videos and extract wholebody poses.

In [ ]:
def process_video(video_path, output_json_path, output_video_path=None):
    """Process a video file to extract wholebody poses.
    
    Args:
        video_path: Path to input video file
        output_json_path: Path to save output pose data
        output_video_path: Optional path to save visualization video
    
    Returns:
        dict containing predicted wholebody poses
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video file: {video_path}")
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Initialize video writer if output path is provided
    video_writer = None
    if output_video_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_writer = cv2.VideoWriter(
            output_video_path,
            fourcc,
            fps,
            (width, height)
        )
    
    # Dictionary to store prediction results
    body_instances_list = []
    left_hand_instances_list = []
    right_hand_instances_list = []
    
    frame_idx = 0
    
    # Process frames
    for _ in tqdm(range(total_frames), desc=f"Processing {os.path.basename(video_path)}"):
        success, frame = cap.read()
        if not success:
            break
        
        # Detect persons in the frame
        person_boxes = detect_person(frame)
        
        # Skip if no person detected
        if len(person_boxes) == 0:
            # Create empty instances for this frame
            body_instances_list.append(dict(frame_id=frame_idx, instances=[]))
            left_hand_instances_list.append(dict(frame_id=frame_idx, instances=[]))
            right_hand_instances_list.append(dict(frame_id=frame_idx, instances=[]))
            
            # Write original frame to video if needed
            if video_writer:
                video_writer.write(frame)
                
            frame_idx += 1
            continue
        
        # Estimate wholebody pose
        body_instances, left_hand_instances, right_hand_instances = estimate_sapiens_pose(
            frame, person_boxes, show=False
        )
        
        # Visualize poses if needed
        if show or video_writer:
            vis_frame = visualize_wholebody_pose(
                frame, body_instances, left_hand_instances, right_hand_instances
            )
            
            if show:
                cv2.imshow('Wholebody Pose', vis_frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            
            if video_writer:
                video_writer.write(vis_frame)
        
        # Convert instances to serializable format
        body_instances_list.append(dict(
            frame_id=frame_idx,
            instances=split_instances(body_instances) if body_instances is not None else []
        ))
        
        left_hand_instances_list.append(dict(
            frame_id=frame_idx,
            instances=split_instances(left_hand_instances) if left_hand_instances is not None else []
        ))
        
        right_hand_instances_list.append(dict(
            frame_id=frame_idx,
            instances=split_instances(right_hand_instances) if right_hand_instances is not None else []
        ))
        
        frame_idx += 1
    
    # Release resources
    cap.release()
    if video_writer:
        video_writer.release()
    if show:
        cv2.destroyAllWindows()
    
    # Save results to JSON file
    if pose_estimator_sapiens is not None:  # Ensure model is initialized
        results = {
            'meta_info': pose_estimator_sapiens.dataset_meta,
            'body_instances': body_instances_list,
            'left_hand_instances': left_hand_instances_list,
            'right_hand_instances': right_hand_instances_list
        }
        
        with open(output_json_path, 'w') as f:
            json.dump(results, f, indent=2)
            
        print(f"Predictions saved to {output_json_path}")
        return results
    else:
        print("SAPIENS model not initialized. No results saved.")
        return None

## Process All Videos in Data Collection

Iterate through all video files in the data collection directory.

In [ ]:
# Get list of video files in data directory
video_files = [f for f in os.listdir(data_dir) if f.endswith('.MP4') or f.endswith('.mp4')]

if not video_files:
    print(f"No video files found in {data_dir}")
else:
    print(f"Found {len(video_files)} video files")
    
    for video_file in video_files:
        video_name = os.path.splitext(video_file)[0]
        video_path = os.path.join(data_dir, video_file)
        
        output_json_path = os.path.join(pose_output_dir, f"{video_name}_wholebody.json")
        output_video_path = os.path.join(vis_output_dir, f"{video_name}_wholebody.mp4")
        
        # Skip if already processed
        if os.path.exists(output_json_path):
            print(f"Skipping {video_name} - already processed")
            continue
            
        print(f"Processing {video_name}...")
        try:
            process_video(video_path, output_json_path, output_video_path)
            print(f"Completed processing {video_name}")
        except Exception as e:
            print(f"Error processing {video_name}: {str(e)}")
            import traceback
            traceback.print_exc()

## Analysis and Visualization

Analyze the detected wholebody poses.

In [ ]:
def visualize_keypoint_distribution(json_path):
    """Visualize the distribution of keypoint confidence scores."""
    if not os.path.exists(json_path):
        print(f"File not found: {json_path}")
        return
    
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Extract confidence scores for each keypoint type
    body_scores = []
    left_hand_scores = []
    right_hand_scores = []
    
    # Process body keypoints
    for frame_data in data['body_instances']:
        for instance in frame_data['instances']:
            if 'keypoint_scores' in instance:
                body_scores.extend(instance['keypoint_scores'])
    
    # Process left hand keypoints
    for frame_data in data['left_hand_instances']:
        for instance in frame_data['instances']:
            if 'keypoint_scores' in instance:
                left_hand_scores.extend(instance['keypoint_scores'])
    
    # Process right hand keypoints
    for frame_data in data['right_hand_instances']:
        for instance in frame_data['instances']:
            if 'keypoint_scores' in instance:
                right_hand_scores.extend(instance['keypoint_scores'])
    
    # Create histogram plots
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.hist(body_scores, bins=20, alpha=0.7)
    plt.title('Body Keypoint Confidence')
    plt.xlabel('Confidence Score')
    plt.ylabel('Count')
    
    plt.subplot(1, 3, 2)
    plt.hist(left_hand_scores, bins=20, alpha=0.7)
    plt.title('Left Hand Keypoint Confidence')
    plt.xlabel('Confidence Score')
    
    plt.subplot(1, 3, 3)
    plt.hist(right_hand_scores, bins=20, alpha=0.7)
    plt.title('Right Hand Keypoint Confidence')
    plt.xlabel('Confidence Score')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"Body keypoints: {len(body_scores)} points, Mean confidence: {np.mean(body_scores):.4f}")
    print(f"Left hand keypoints: {len(left_hand_scores)} points, Mean confidence: {np.mean(left_hand_scores) if left_hand_scores else 0:.4f}")
    print(f"Right hand keypoints: {len(right_hand_scores)} points, Mean confidence: {np.mean(right_hand_scores) if right_hand_scores else 0:.4f}")

In [ ]:
# Analyze the first processed video file (if any)
json_files = [f for f in os.listdir(pose_output_dir) if f.endswith('_wholebody.json')]

if json_files:
    # Select the first JSON file for analysis
    json_path = os.path.join(pose_output_dir, json_files[0])
    print(f"Analyzing {json_files[0]}...")
    visualize_keypoint_distribution(json_path)
else:
    print("No processed files found for analysis.")

## Conclusion

The SAPIENS model provides comprehensive wholebody pose estimation by detecting body, hand, and face keypoints in a single inference pass. This approach offers several advantages over using separate models for different body parts:

1. **Efficiency**: A single model means less compute overhead compared to running multiple models
2. **Consistency**: The model maintains anatomical consistency between body and hand poses
3. **Completeness**: Captures full body pose information in a single prediction

The estimated poses can be used for:
- 3D pose reconstruction
- Action recognition
- Gesture analysis
- Motion capture